# Módulo 4 · Clase 3 — Consultas avanzadas: JOIN, GROUP BY y subconsultas

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema | Para qué sirve |
|---|------|----------------|
| 1 | `INNER JOIN` | Combinar filas que tienen coincidencias en ambas tablas |
| 2 | `LEFT JOIN` | Incluir todas las filas de la tabla izquierda |
| 3 | `RIGHT JOIN` / `FULL JOIN` | Variantes de JOIN externo |
| 4 | `GROUP BY` | Agrupar filas y agregar por grupo |
| 5 | `HAVING` | Filtrar grupos (como WHERE pero para grupos) |
| 6 | Subconsultas | Anidar un SELECT dentro de otro |
| 7 | Práctica | Análisis de ventas de la tienda |

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

conn = sqlite3.connect(Path.home() / "tienda.db")

def q(sql, *params):
    return pd.read_sql_query(sql, conn, params=params)

# Vistazo rápido a las tablas disponibles
print("Tablas:", q("SELECT name FROM sqlite_master WHERE type='table'")['name'].tolist())

---
## 1. El concepto de JOIN

Un `JOIN` combina columnas de dos (o más) tablas relacionadas mediante una condición.

```
clientes                          pedidos
┌────┬──────────┬────────┐        ┌─────┬────────────┬───────────┐
│ id │ nombre   │ ciudad │        │ id  │ cliente_id │ estado    │
├────┼──────────┼────────┤        ├─────┼────────────┼───────────┤
│  1 │ Ana      │ Madrid │◄──────►│ 101 │     1      │ entregado │
│  2 │ Carlos   │ BCN    │◄──────►│ 103 │     2      │ entregado │
│  3 │ Lucía    │ Sevilla│        │ 104 │     3      │ entregado │
└────┴──────────┴────────┘        └─────┴────────────┴───────────┘

JOIN ON clientes.id = pedidos.cliente_id

┌────────────┬──────────────────────────────┐
│ c.nombre   │ p.id   p.estado              │
├────────────┼──────────────────────────────┤
│ Ana        │ 101    entregado             │
│ Carlos     │ 103    entregado             │
│ Lucía      │ 104    entregado             │
└────────────┴──────────────────────────────┘
```

**Alias de tabla**: cuando usas JOIN es habitual dar alias cortos a las tablas para no repetir su nombre completo:
```sql
FROM clientes c
JOIN pedidos p ON c.id = p.cliente_id
```

---
## 1. `INNER JOIN`

Devuelve solo las filas que tienen coincidencia **en ambas tablas**. Si una fila de la tabla A no tiene ninguna correspondiente en B, no aparece en el resultado.

```sql
SELECT col1, col2
FROM   tabla_a a
INNER JOIN tabla_b b ON a.fk = b.id;

-- INNER es la palabra clave por defecto; estos dos son equivalentes:
FROM tabla_a JOIN tabla_b ON ...
FROM tabla_a INNER JOIN tabla_b ON ...
```

In [ ]:
# Pedidos con nombre del cliente
print("=== Pedidos con nombre del cliente ===")
display(q("""
    SELECT p.id          AS pedido_id,
           c.nombre      AS cliente,
           c.ciudad,
           p.fecha,
           p.estado
    FROM   pedidos   p
    JOIN   clientes  c ON p.cliente_id = c.id
    ORDER  BY p.fecha
"""))

# Productos con nombre de categoría
print("\n=== Productos con categoría ===")
display(q("""
    SELECT prod.nombre   AS producto,
           cat.nombre    AS categoria,
           prod.precio,
           prod.stock
    FROM   productos  prod
    JOIN   categorias cat ON prod.categoria_id = cat.id
    ORDER  BY cat.nombre, prod.precio DESC
"""))

# JOIN de tres tablas: líneas de pedido con todo
print("\n=== Detalle completo de pedidos ===")
display(q("""
    SELECT p.id           AS pedido,
           c.nombre       AS cliente,
           prod.nombre    AS producto,
           pp.cantidad,
           pp.precio_ud,
           ROUND(pp.cantidad * pp.precio_ud, 2) AS subtotal
    FROM   pedido_productos pp
    JOIN   pedidos   p    ON pp.pedido_id   = p.id
    JOIN   clientes  c    ON p.cliente_id   = c.id
    JOIN   productos prod ON pp.producto_id = prod.id
    ORDER  BY p.id, prod.nombre
"""))

---
## 2. `LEFT JOIN`

Devuelve **todas las filas de la tabla izquierda** (la del `FROM`), y las columnas de la tabla derecha rellenadas con `NULL` cuando no hay coincidencia.

```
clientes (A)    pedidos (B)        LEFT JOIN resultado
┌────┬──────┐   ┌─────┬──────┐    ┌──────┬─────────┐
│  1 │ Ana  │   │ 101 │  1   │    │ Ana  │ 101     │
│  2 │ Bob  │   │ 102 │  1   │    │ Ana  │ 102     │
│  3 │ Cara │   │ 103 │  2   │    │ Bob  │ 103     │
└────┴──────┘   └─────┴──────┘    │ Cara │ NULL    │ ← sin pedidos
                                   └──────┴─────────┘
```

Caso de uso principal: **encontrar filas sin correspondencia** (`WHERE b.id IS NULL`).

In [ ]:
# Clientes Y su número de pedidos (incluyendo los que nunca han pedido)
print("=== Clientes con número de pedidos (LEFT JOIN) ===")
display(q("""
    SELECT c.nombre,
           c.ciudad,
           COUNT(p.id) AS num_pedidos
    FROM   clientes c
    LEFT JOIN pedidos p ON c.id = p.cliente_id
    GROUP  BY c.id, c.nombre, c.ciudad
    ORDER  BY num_pedidos DESC
"""))

# Clientes que NUNCA han pedido
print("\n=== Clientes sin ningún pedido ===")
display(q("""
    SELECT c.nombre, c.email
    FROM   clientes c
    LEFT JOIN pedidos p ON c.id = p.cliente_id
    WHERE  p.id IS NULL
"""))

# Productos que nunca han sido pedidos
print("\n=== Productos que nunca se han vendido ===")
display(q("""
    SELECT prod.nombre, prod.precio, prod.stock
    FROM   productos prod
    LEFT JOIN pedido_productos pp ON prod.id = pp.producto_id
    WHERE  pp.producto_id IS NULL
"""))

---
## 3. `GROUP BY` — Agrupar y agregar

```sql
SELECT   col_agrupacion, AGG(col)
FROM     tabla
[JOIN ...]
[WHERE  condición_de_fila]
GROUP BY col_agrupacion
[HAVING condición_de_grupo]
ORDER BY ...
```

**Regla clave**: en un `SELECT` con `GROUP BY`, cada columna del `SELECT` debe ser:
- Una de las columnas agrupadas, **o**
- El argumento de una función de agregación.

```
ANTES del GROUP BY — filas individuales:
ciudad      precio
Madrid      599.99
Madrid       45.00
Barcelona   350.00

DESPUÉS del GROUP BY ciudad:
ciudad      COUNT(*)   AVG(precio)
Madrid        2          322.50
Barcelona     1          350.00
```

In [ ]:
# Número de productos por categoría
print("=== Productos por categoría ===")
display(q("""
    SELECT cat.nombre              AS categoria,
           COUNT(prod.id)          AS num_productos,
           ROUND(AVG(prod.precio), 2) AS precio_medio,
           SUM(prod.stock)         AS stock_total
    FROM   categorias cat
    JOIN   productos  prod ON prod.categoria_id = cat.id
    GROUP  BY cat.id, cat.nombre
    ORDER  BY num_productos DESC
"""))

# Facturación por pedido (total de cada pedido)
print("\n=== Total por pedido ===")
display(q("""
    SELECT p.id              AS pedido_id,
           c.nombre          AS cliente,
           p.fecha,
           p.estado,
           COUNT(pp.producto_id)                      AS num_lineas,
           ROUND(SUM(pp.cantidad * pp.precio_ud), 2)  AS total
    FROM   pedidos          p
    JOIN   clientes         c  ON p.cliente_id   = c.id
    JOIN   pedido_productos pp ON p.id           = pp.pedido_id
    GROUP  BY p.id, c.nombre, p.fecha, p.estado
    ORDER  BY total DESC
"""))

# Pedidos y gasto por cliente
print("\n=== Gasto total por cliente ===")
display(q("""
    SELECT c.nombre                                    AS cliente,
           COUNT(DISTINCT p.id)                        AS pedidos,
           ROUND(SUM(pp.cantidad * pp.precio_ud), 2)   AS gasto_total
    FROM   clientes         c
    JOIN   pedidos          p  ON c.id          = p.cliente_id
    JOIN   pedido_productos pp ON p.id          = pp.pedido_id
    GROUP  BY c.id, c.nombre
    ORDER  BY gasto_total DESC
"""))

---
## 4. `HAVING` — Filtrar grupos

```
WHERE  → filtra filas ANTES del agrupamiento
HAVING → filtra grupos DESPUÉS del agrupamiento
```

```sql
SELECT categoria_id, AVG(precio) AS media
FROM productos
WHERE stock > 0           -- elimina productos sin stock antes de agrupar
GROUP BY categoria_id
HAVING AVG(precio) > 100  -- solo categorías con precio medio > 100
```

In [ ]:
# Clientes con más de un pedido
print("=== Clientes con más de 1 pedido ===")
display(q("""
    SELECT c.nombre, COUNT(p.id) AS pedidos
    FROM   clientes c
    JOIN   pedidos  p ON c.id = p.cliente_id
    GROUP  BY c.id, c.nombre
    HAVING COUNT(p.id) > 1
    ORDER  BY pedidos DESC
"""))

# Categorías cuyo precio medio supera 200€
print("\n=== Categorías con precio medio > 200€ ===")
display(q("""
    SELECT cat.nombre                  AS categoria,
           ROUND(AVG(prod.precio), 2)  AS precio_medio
    FROM   categorias cat
    JOIN   productos  prod ON prod.categoria_id = cat.id
    GROUP  BY cat.id, cat.nombre
    HAVING AVG(prod.precio) > 200
    ORDER  BY precio_medio DESC
"""))

# Productos más vendidos (más de 1 unidad en total)
print("\n=== Productos vendidos en >1 unidad total ===")
display(q("""
    SELECT prod.nombre,
           SUM(pp.cantidad) AS unidades_vendidas
    FROM   productos        prod
    JOIN   pedido_productos pp ON prod.id = pp.producto_id
    GROUP  BY prod.id, prod.nombre
    HAVING SUM(pp.cantidad) > 1
    ORDER  BY unidades_vendidas DESC
"""))

---
## 5. Subconsultas

Una subconsulta es un `SELECT` anidado dentro de otro `SELECT`.

```sql
-- En WHERE: comparar con un valor calculado
SELECT nombre, precio
FROM productos
WHERE precio > (SELECT AVG(precio) FROM productos);

-- En WHERE con IN: filtrar por una lista
SELECT nombre FROM clientes
WHERE id IN (SELECT cliente_id FROM pedidos WHERE estado = 'pendiente');

-- En FROM: tratar el resultado como tabla temporal
SELECT cat, media
FROM (SELECT categoria_id AS cat, AVG(precio) AS media
      FROM productos GROUP BY categoria_id) sub
WHERE media > 100;
```

In [ ]:
# Productos por encima del precio medio
print("=== Productos más caros que la media ===")
display(q("""
    SELECT nombre,
           precio,
           ROUND((SELECT AVG(precio) FROM productos), 2) AS precio_medio
    FROM productos
    WHERE precio > (SELECT AVG(precio) FROM productos)
    ORDER BY precio DESC
"""))

# Clientes con pedidos pendientes
print("\n=== Clientes con pedidos pendientes ===")
display(q("""
    SELECT nombre, email
    FROM   clientes
    WHERE  id IN (
        SELECT DISTINCT cliente_id
        FROM   pedidos
        WHERE  estado = 'pendiente'
    )
"""))

# El pedido más caro de cada cliente
print("\n=== Pedido de mayor importe por cliente ===")
display(q("""
    SELECT c.nombre AS cliente,
           sub.pedido_id,
           sub.total
    FROM (
        SELECT p.cliente_id,
               p.id AS pedido_id,
               ROUND(SUM(pp.cantidad * pp.precio_ud), 2) AS total
        FROM   pedidos          p
        JOIN   pedido_productos pp ON p.id = pp.pedido_id
        GROUP  BY p.id, p.cliente_id
    ) sub
    JOIN clientes c ON sub.cliente_id = c.id
    WHERE sub.total = (
        SELECT MAX(sub2.total)
        FROM (
            SELECT p2.cliente_id,
                   SUM(pp2.cantidad * pp2.precio_ud) AS total
            FROM   pedidos          p2
            JOIN   pedido_productos pp2 ON p2.id = pp2.pedido_id
            GROUP  BY p2.id, p2.cliente_id
        ) sub2
        WHERE sub2.cliente_id = sub.cliente_id
    )
    ORDER BY sub.total DESC
"""))

---
## 6. Práctica — Análisis de ventas

---

### Ejercicio 1 — JOIN básico

**Enunciado:** Escribe una consulta que devuelva:
- Nombre del producto, nombre de la categoría, precio y stock.
- Solo productos de la categoría «Informática».
- Ordenados por precio descendente.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
display(q("""
    SELECT prod.nombre      AS producto,
           cat.nombre       AS categoria,
           prod.precio,
           prod.stock
    FROM   productos  prod
    JOIN   categorias cat ON prod.categoria_id = cat.id
    WHERE  cat.nombre = 'Informática'
    ORDER  BY prod.precio DESC
"""))
```
</details>

---
### Ejercicio 2 — GROUP BY

**Enunciado:** Calcula, por estado de pedido (`pendiente`, `enviado`, `entregado`):
- Número de pedidos.
- Importe total acumulado.
- Importe medio por pedido.

Ordena por importe total descendente.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
display(q("""
    SELECT p.estado,
           COUNT(DISTINCT p.id)                        AS num_pedidos,
           ROUND(SUM(pp.cantidad * pp.precio_ud), 2)   AS total_acumulado,
           ROUND(AVG(pp.cantidad * pp.precio_ud), 2)   AS media_linea
    FROM   pedidos          p
    JOIN   pedido_productos pp ON p.id = pp.pedido_id
    GROUP  BY p.estado
    ORDER  BY total_acumulado DESC
"""))
```
</details>

---
### Ejercicio 3 — HAVING

**Enunciado:** Encuentra los productos que han generado más de 200€ en ventas totales. Muestra nombre, categoría, unidades vendidas e importe total. Ordena por importe descendente.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
display(q("""
    SELECT prod.nombre                                 AS producto,
           cat.nombre                                  AS categoria,
           SUM(pp.cantidad)                            AS unidades_vendidas,
           ROUND(SUM(pp.cantidad * pp.precio_ud), 2)   AS total_ventas
    FROM   productos        prod
    JOIN   categorias       cat ON prod.categoria_id = cat.id
    JOIN   pedido_productos pp  ON prod.id           = pp.producto_id
    GROUP  BY prod.id, prod.nombre, cat.nombre
    HAVING SUM(pp.cantidad * pp.precio_ud) > 200
    ORDER  BY total_ventas DESC
"""))
```
</details>

---
### Ejercicio 4 — LEFT JOIN para detectar huecos

**Enunciado:**
1. Lista las categorías que no tienen ningún producto.
2. Lista los clientes que tienen pedidos pero todos están en estado «pendiente» (nunca han recibido un pedido).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
# 1. Categorías sin productos
display(q("""
    SELECT cat.nombre AS categoria
    FROM   categorias cat
    LEFT JOIN productos prod ON prod.categoria_id = cat.id
    WHERE  prod.id IS NULL
"""))

# 2. Clientes cuyos pedidos son todos 'pendiente'
display(q("""
    SELECT c.nombre, COUNT(p.id) AS pedidos_pendientes
    FROM   clientes c
    JOIN   pedidos  p ON c.id = p.cliente_id
    GROUP  BY c.id, c.nombre
    HAVING COUNT(p.id) = SUM(p.estado = 'pendiente')
"""))
```
</details>

---
### Ejercicio 5 (reto) — Función `ranking_ventas`

**Enunciado:** Escribe la función `ranking_ventas(conn, top_n=5)` que devuelva un DataFrame con el ranking de los `top_n` productos más vendidos (por importe total), incluyendo:
- Posición en el ranking.
- Nombre del producto y categoría.
- Unidades vendidas, importe total y precio medio de venta.
- Cuántos pedidos distintos incluyeron ese producto.
- Si el producto tiene stock suficiente para cubrir la demanda media (stock >= unidades_vendidas / numero_pedidos).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import pandas as pd

def ranking_ventas(conn, top_n=5):
    df = pd.read_sql_query(f"""
        SELECT prod.nombre                                  AS producto,
               cat.nombre                                   AS categoria,
               SUM(pp.cantidad)                             AS unidades,
               ROUND(SUM(pp.cantidad * pp.precio_ud), 2)    AS importe_total,
               ROUND(AVG(pp.precio_ud), 2)                  AS precio_medio_venta,
               COUNT(DISTINCT pp.pedido_id)                 AS num_pedidos,
               prod.stock
        FROM   productos        prod
        JOIN   categorias       cat ON prod.categoria_id = cat.id
        JOIN   pedido_productos pp  ON prod.id           = pp.producto_id
        GROUP  BY prod.id, prod.nombre, cat.nombre, prod.stock
        ORDER  BY importe_total DESC
        LIMIT  {top_n}
    """, conn)

    df.insert(0, 'ranking', range(1, len(df) + 1))
    df['demanda_media'] = (df['unidades'] / df['num_pedidos']).round(1)
    df['stock_ok'] = df['stock'] >= df['demanda_media']
    df['stock_ok'] = df['stock_ok'].map({True: '✅', False: '⚠️'})
    return df.drop(columns=['stock'])

display(ranking_ventas(conn, top_n=5))
```
</details>

---
## Resumen de la clase

```sql
-- Tipos de JOIN
FROM a INNER JOIN b ON a.id = b.fk    -- solo coincidencias
FROM a LEFT  JOIN b ON a.id = b.fk    -- todas las filas de a

-- Orden de evaluación
FROM → JOIN → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT

-- GROUP BY + HAVING
SELECT col, COUNT(*), SUM(x)
FROM tabla
WHERE condicion_de_fila          -- filtra ANTES de agrupar
GROUP BY col
HAVING COUNT(*) > 3              -- filtra DESPUÉS de agrupar
ORDER BY SUM(x) DESC;

-- Subconsulta escalar
WHERE precio > (SELECT AVG(precio) FROM productos)

-- Subconsulta con IN
WHERE id IN (SELECT cliente_id FROM pedidos WHERE estado = 'pendiente')
```

## ¿Qué viene en la próxima clase?

En la **Clase 4** aprenderemos a **diseñar** bases de datos: normalización (1FN, 2FN, 3FN) y tipos de relaciones (1:1, 1:N, N:M).

In [ ]:
conn.close()
print("Conexión cerrada")